### Messages


Messages are the fundamental unit of context for models in LancChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting witn an LLM. Messages are objects that contain:

   Role - Identifies the message type (e.g system, user)
   Content - Represents the actual content of the message (like text, images, audio, documents, etc)
   Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called. 

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


model = init_chat_model("groq:qwen/qwen3-32b")

### Text Prompts

Text prompts are strings - ideal for straightforward generation tasks where you don't need to retain conversation hnistory.

In [ ]:
model.invoke("What is LangChain?")

Use text prompts when:
  You have a single, standalone request
  You don't need conversation history
  You want minimal code complexity

## Message Prompts

Alternatively, you can pass in a list of messages to the model by providing a list message objects.

Message types
  System Message - Tells the model how to behave and provide context for interactions
  Human Message  - Represents user input and interactions with the model
  AI Message     - Responses generated by the model, including text content, tool calls, and metadata
  Tool Message   - Represents the outputs of tool calls

### System Message

A System message represnets an initial set of instructions that primes the model's behavior. You can use a system message to set the tone, define the model's role, and establish guidelines for responses.

### Human Message

A Human message represents user input and interactions. They can contain text, images, audio, files and any other amount of multimodal content.

### AI Message

An AI message represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

### Tool Message

For models that support tool calling. AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.


In [4]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on artificial intelligence")
]

response = model.invoke(messages)
response.content

'<think>\nOkay, the user wants me to write a poem about artificial intelligence. Let me start by thinking about the key themes related to AI. There\'s the creation aspect, like how AI is man-made. Maybe use metaphors like circuits or code. Then there\'s the idea of learning and growth, so words like "nurtured" or "taught" could work.\n\nI should also touch on the dual nature of AI—its potential for good and the risks. Words like "guardian" and "shadow" might contrast that. Need to mention different applications, like medicine or art, to show versatility. \n\nRhythm and rhyme scheme are important. Maybe a consistent structure with quatrains. Let me brainstorm some lines. Start with creation: "In circuits woven, thoughts are born," then touch on learning. \n\nNeed to address ethical concerns without being too negative. Use terms like "guardian" and "shadow" to imply responsibility. End on a hopeful note, emphasizing collaboration between humans and AI. Maybe something like "hand in hand"

In [ ]:
system_msg = SystemMessage("You are a helpful coding assistant") 
messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]

respone = model.invoke(messages)
print(respone.content)

In [ ]:
## Detailed info to the LLM through the System Message
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do i create a REST API?")
]

respone = model.invoke(messages)
print(respone.content)

In [ ]:
## Message Metadata
human_msg = HumanMessage(
    content = "Hello!",
    name = "alice", # Optional: identify different users
    id = "msg_123"  # Optional: unique identifier for tracing 
)

respone = model.invoke([human_msg])
print(respone.content)

In [ ]:
from langchain.messages import AIMessage, HumanMessage, SystemMessage

# Create an AI message manually (e.g, for conversation history)
ai_msg = AIMessage("I'd be happy to gelp you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful Assistant"),
    HumanMessage("Can you help me?"),
    ai_msg, # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]
respone = model.invoke(messages)
print(respone.content)

In [ ]:
from langchain.messages import AIMessage, ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the message for brevity)

ai_message = AIMessage(
    content=[],
    tool_calls = [{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72Deg F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id = "call_123" # Must match the call ID
)

# Continue Conversation
messages= [
    HumanMessage("What's the weather in San Francisco?")
    ai_msg, # Model's tool call
    tool_message, # Tool execution result 
]

response = model.invoke(messages) # Model processes the result